In [14]:
#For binding db extraction
import requests
import csv

def fetch_ic50_values(uniprot_id):
    # Construct the URL for the API request
    url = f"http://bindingdb.org/rest/getLigandsByUniprots?uniprot={uniprot_id}"
    
    try:
        # Send the GET request to the BindingDB API
        response = requests.get(url)
        response.raise_for_status()  # Raise an error for bad responses
        
        # Parse the JSON response
        data = response.json()
        
        # Drill down to the list of affinity dictionaries
        affinities = data.get('getLindsByUniprotsResponse', {}) \
                         .get('affinities', [])
        
        # Print the first few entries for debugging
        print(f"Found {len(affinities)} ligands for {uniprot_id}")
        for entry in affinities[:3]:
            print(entry)
        
        # Save the data to a CSV file
        csv_filename = f"{uniprot_id}_ic50_values.csv"
        with open(csv_filename, mode='w', newline='') as csv_file:
            writer = csv.writer(csv_file)
            
            # Write the header
            writer.writerow([
                'Monomer ID',
                'SMILES',
                'Affinity Type',
                'Affinity Value',
                'PMID',
                'DOI'
            ])
            
            # Write each ligand's data
            for ligand in affinities:
                writer.writerow([
                    ligand.get('monomerid', 'N/A'),
                    ligand.get('smile',    'N/A'),
                    ligand.get('affinity_type', 'N/A'),
                    ligand.get('affinity',      'N/A'),
                    ligand.get('pmid',          'N/A'),
                    ligand.get('doi',           'N/A'),
                ])
        
        print(f"Data successfully saved to {csv_filename}")
    
    except requests.exceptions.RequestException as e:
        print(f"An error occurred while fetching data: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# Example usage
if __name__ == "__main__":
    fetch_ic50_values("O15264")


Found 977 ligands for O15264
{'query': 'O15264', 'monomerid': '50314776', 'smile': 'Cc1ccc(cc1-c1cc2cnn(-c3c(F)cccc3F)c2n(C)c1=O)C(=O)NC1CC1', 'affinity_type': 'Ki', 'affinity': '2100', 'pmid': '20218619', 'doi': '10.1021/jm100095x'}
{'query': 'O15264', 'monomerid': '50224883', 'smile': 'Clc1cc2NC(=O)Nc3cnc(C#N)c(OCCCCOc2cc1NCc1cncs1)n3', 'affinity_type': 'Ki', 'affinity': '6812', 'pmid': '17935989', 'doi': '10.1016/j.bmcl.2007.09.063'}
{'query': 'O15264', 'monomerid': '50402020', 'smile': 'CC(C)(C)Nc1c(Nc2ccnc(Nc3ccc(cc3)-c3ccncc3)n2)c(=O)c1=O', 'affinity_type': 'Ki', 'affinity': '5900', 'pmid': '23103095', 'doi': '10.1016/j.bmcl.2012.10.009'}
Data successfully saved to O15264_ic50_values.csv


In [15]:
# For chembl dtaa extraction using API calling 
import requests
import csv

BASE_URL = "https://www.ebi.ac.uk/chembl/api/data/activity.json"

def fetch_ic50_for_target(target_id, output_csv="TTBK1_IC50_chembl.csv"):
    params = {
        "target_chembl_id": target_id,   # ← use target filter
        "standard_type":    "IC50",
        "limit":            1000,
        "offset":           0
    }
    all_activities = []

    while True:
        resp = requests.get(BASE_URL, params=params)
        resp.raise_for_status()
        data = resp.json()
        acts = data.get("activities", [])
        if not acts:
            break
        all_activities.extend(acts)
        if len(acts) < params["limit"]:
            break
        params["offset"] += params["limit"]

    # write to CSV
    with open(output_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "activity_id",
            "assay_chembl_id",
            "molecule_chembl_id",
            "standard_type",
            "standard_value",
            "standard_units",
            "pchembl_value",
            "document_chembl_id"
        ])
        for a in all_activities:
            writer.writerow([
                a.get("activity_id"),
                a.get("assay_chembl_id"),
                a.get("molecule_chembl_id"),
                a.get("standard_type"),
                a.get("standard_value"),
                a.get("standard_units"),
                a.get("pchembl_value"),
                a.get("document_chembl_id"),
            ])

    print(f"Fetched {len(all_activities)} IC₅₀ records for target {target_id} into {output_csv}")

if __name__ == "__main__":
    # CHEMBL1926492 is the TTBK2 target
    fetch_ic50_for_target("CHEMBL2939", output_csv="MAPK13_CHEMBL2939_IC50_chembl.csv")


Fetched 108 IC₅₀ records for target CHEMBL2939 into MAPK13_CHEMBL2939_IC50_chembl.csv


In [24]:
import os
os.getcwd()
#cd= r"/media/umar/New Volume/Shareable folders/Data_TDC/data_IC50_chemberta"
#os.chdir(cd)
#os.getcwd()

'/media/umar/New Volume/Shareable folders/Data_TDC/data_IC50_chemberta'

In [ ]:
import json
import pandas as pd
#Go to https://www.ebi.ac.uk/chembl/api/data/activity.json?target_chembl_id=CHEMBL1926492&standard_type=IC50 in your browser.
# Load your manually saved file
with open("CHEMBL1926492_IC50.json") as f:
    data = json.load(f)

# Normalize and display
records = data.get("activities", [])
df = pd.json_normalize(records)
cols = [
    "activity_id", "assay_chembl_id", "molecule_chembl_id", "target_chembl_id",
    "standard_type", "standard_value", "standard_units", "pchembl_value", "document_chembl_id"
]
df = df.reindex(columns=cols)
df.head()
df.to_csv("TTBK1_chembl_IC50.csv")


In [ ]:
#For mapping smiles to id from chembl
import pandas as pd
import requests
from time import sleep
from tqdm import tqdm

# Load your data
df = pd.read_csv("ABL1_CHEMBL1862_IC50_chembl.csv")  # or pd.read_excel("your_data.xlsx")

# Ensure column is named correctly
chembl_ids = df['molecule_chembl_id'].unique()

# Function to fetch SMILES
def get_smiles(chembl_id):
    url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/{chembl_id}.json"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            return data.get('molecule_structures', {}).get('canonical_smiles', None)
    except Exception as e:
        return None

# Map all unique ChEMBL IDs to SMILES
smiles_map = {}
for cid in tqdm(chembl_ids, desc="Fetching SMILES"):
    smiles_map[cid] = get_smiles(cid)
    sleep(0.1)  # Be kind to the API

# Map to original dataframe
df['smiles'] = df['molecule_chembl_id'].map(smiles_map)

# Reorder columns: insert SMILES after the first column (index 1)
cols = df.columns.tolist()
cols.insert(1, cols.pop(cols.index('smiles')))
df = df[cols] 

# Save the new dataset
df.to_csv("ABL1_CHEMBL1862_IC50_chembl_Wsmile.csv", index=False)
print("✅ SMILES added and file saved as 'ABL1_CHEMBL1862_IC50_chembl.csv'")


Fetching SMILES:   0%|          | 0/2173 [00:00<?, ?it/s]

Fetching SMILES: 100%|██████████| 2173/2173 [33:12<00:00,  1.09it/s]  

✅ SMILES added and file saved as 'ABL1_CHEMBL1862_IC50_chembl.csv'


In [3]:
import os
os.getcwd()

'/media/umar/New Volume/Shareable folders/Data_TDC/data_IC50_chemberta/ML_MT'